# 01: Feature extraction

A multiple sequence alignment (MSA) contains the query protein and related protein sequences aligned by position. It reveals which residues are conserved or vary together. This notebook converts an A3M alignment into tensors the model can process.

**Read alongside:** `../src/af2_from_scratch/feature_extraction.py`


## Stage map

```text
Related sequences (A3M)
          |
          v
Parse sequences and count insertions
          |
          v
Remove duplicates and encode amino acids
          |
          v
Calculate amino-acid frequencies by position
          |
          v
Select and mask sequences for this model pass
          |
          v
msa_feat | extra_msa_feat | target_feat | residue_index
```

`msa_features(...)` parses reusable features once. `sample_batch(...)` selects and masks a fresh model input for each training step.


In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")  # package source lives one level up
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (8, 4)

## 1. Look at the raw file

Lines beginning with `>` identify sequences. The first sequence is the **query**. Uppercase letters are aligned residues, lowercase letters are insertions relative to the query, and `-` represents a gap.


In [ ]:
lines = open("../examples/tautomerase/alignment.a3m").readlines()
print("".join(lines[:4]))

## 2. Parse one sequence

`parse_seq` removes lowercase insertions so every sequence stays aligned to the query. It scans from left to right, counts lowercase residues, assigns that count to the next aligned residue, and then resets the count to zero.

The output keeps AlphaFold's name `deletion_matrix`, although each value records the lowercase insertions immediately before that aligned position.


In [ ]:
from af2_from_scratch.feature_extraction import parse_seq

raw = "PIAxqIHsdILEGR"  # xq and sd are insertion runs
clean, deletion_counts = parse_seq(raw)
print("raw:       ", raw)
print("clean:     ", clean)
print("insertions:", deletion_counts)  # [0, 0, 0, 2, 0, 2, 0, 0, 0, 0]

## 3. Run the full extraction

`msa_features` parses 8,361 sequences, removes duplicates, and encodes each amino acid as a one-hot vector. It then calculates the **MSA profile**: the frequency of each amino acid at every residue position.


In [ ]:
from af2_from_scratch.feature_extraction import msa_features

f = msa_features("../examples/tautomerase/alignment.a3m")
for k, v in f.items():
    print(f"{k:16s} {tuple(v.shape)}")

The **MSA profile** summarizes which amino acids appear at each position. A position dominated by one amino acid is highly conserved, which can indicate structural or functional constraints.


In [ ]:
plt.imshow(f["profile"][:, :20].T, aspect="auto", cmap="viridis")
plt.xlabel("residue position")
plt.ylabel("amino acid (A..V)")
plt.title("MSA profile: evolution's per-position fingerprint")
plt.colorbar()
plt.show()

## 4. Build a training batch

`sample_batch` randomly selects rows for the main and extra MSAs. It masks 15% of positions in the main MSA so the model must infer information from sequence context.

Of the selected positions:

- 70% become a zero mask
- 10% become a random amino acid
- 10% use the MSA profile
- 10% remain unchanged


In [ ]:
from af2_from_scratch.feature_extraction import sample_batch

b = sample_batch(f, n_clu=128, n_ext=128, seed=0)
for k, v in b.items():
    print(f"{k:16s} {tuple(v.shape)}")
print("\nmsa_feat 45 = masked one-hot(22) + deletion(1) + profile(22)")
print("extra   23 = one-hot(22) + deletion(1)   (no profile -> simpler)")

**Recap:** A3M text → aligned sequences and insertion counts → numerical features → a sampled and masked model batch.

Next, `02_feature_embedding.ipynb` converts these features into the MSA, residue-pair, and extra-MSA representations used by the model.
